# Exploratory Data Analysis: PubMed RCT 20k Dataset

This notebook performs comprehensive exploratory data analysis on the PubMed RCT 20k dataset,
which contains sentences from medical abstracts labeled with their rhetorical roles:
**BACKGROUND**, **OBJECTIVE**, **METHODS**, **RESULTS**, and **CONCLUSIONS**.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Dataset

In [ ]:
dataset = load_dataset('armanc/pubmed-rct20k')
print('Splits:', list(dataset.keys()))
for split in dataset:
    print(f'  {split}: {len(dataset[split]):,} examples')

In [ ]:
# Convert to DataFrames for easier analysis
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

print('Columns:', train_df.columns.tolist())
train_df.head(10)

## 2. Label Distribution Analysis

In [ ]:
label_col = 'label'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    counts = df[label_col].value_counts().sort_index()
    colors = sns.color_palette('Set2', n_colors=len(counts))
    counts.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'{name} Set Label Distribution (n={len(df):,})')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    for i, v in enumerate(counts):
        ax.text(i, v + 50, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../results/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Label proportions
print('Label proportions (Train):')
print(train_df[label_col].value_counts(normalize=True).sort_index().apply(lambda x: f'{x:.2%}'))
print()
print('Label proportions (Test):')
print(test_df[label_col].value_counts(normalize=True).sort_index().apply(lambda x: f'{x:.2%}'))

## 3. Sentence Length Analysis

In [ ]:
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
train_df['char_count'] = train_df['text'].apply(len)

print('Word count statistics:')
print(train_df['word_count'].describe())
print(f'\n99th percentile: {train_df["word_count"].quantile(0.99):.0f} words')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall word count distribution
axes[0].hist(train_df['word_count'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(train_df['word_count'].median(), color='red', linestyle='--', label=f"Median={train_df['word_count'].median():.0f}")
axes[0].axvline(128, color='orange', linestyle='--', label='Max length=128')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sentence Word Count Distribution')
axes[0].legend()

# Word count by label
labels_unique = sorted(train_df[label_col].unique())
data_by_label = [train_df[train_df[label_col] == l]['word_count'].values for l in labels_unique]
bp = axes[1].boxplot(data_by_label, labels=labels_unique, patch_artist=True)
colors = sns.color_palette('Set2', n_colors=len(labels_unique))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Word Count')
axes[1].set_title('Word Count by Label')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/sentence_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Vocabulary Analysis

In [ ]:
# Word frequency analysis
all_words = ' '.join(train_df['text'].str.lower()).split()
word_freq = Counter(all_words)

print(f'Total tokens: {len(all_words):,}')
print(f'Unique tokens: {len(word_freq):,}')
print(f'\nTop 20 most common words:')
for word, count in word_freq.most_common(20):
    print(f'  {word}: {count:,}')

In [ ]:
# Word frequency distribution (log scale)
freq_values = sorted(word_freq.values(), reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(freq_values)+1), freq_values)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Word Rank (log)')
axes[0].set_ylabel('Frequency (log)')
axes[0].set_title('Zipf\'s Law - Word Frequency Distribution')

# Vocabulary coverage vs min_freq threshold
thresholds = [1, 2, 3, 5, 10, 20, 50]
vocab_sizes = [sum(1 for c in word_freq.values() if c >= t) for t in thresholds]
axes[1].bar(range(len(thresholds)), vocab_sizes, color='steelblue')
axes[1].set_xticks(range(len(thresholds)))
axes[1].set_xticklabels([str(t) for t in thresholds])
axes[1].set_xlabel('Minimum Frequency Threshold')
axes[1].set_ylabel('Vocabulary Size')
axes[1].set_title('Vocabulary Size vs. Min Frequency')
for i, v in enumerate(vocab_sizes):
    axes[1].text(i, v + 200, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../results/vocabulary_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Label-Specific Word Analysis

In [ ]:
from collections import defaultdict
import re

# Common stopwords to filter out
stopwords = {'the', 'of', 'and', 'to', 'a', 'in', 'was', 'were', 'is', 'are', 'for',
             'with', 'that', 'this', 'an', 'be', 'as', 'on', 'by', 'or', 'at', 'from',
             'it', 'not', 'has', 'had', 'have', 'been', 'we', 'our', 'than', 'its',
             'which', 'their', 'but', 'can', 'may', 'more', 'no', 'all', 'between'}

label_words = defaultdict(Counter)
for _, row in train_df.iterrows():
    words = re.findall(r'\b[a-z]+\b', row['text'].lower())
    words = [w for w in words if w not in stopwords and len(w) > 2]
    label_words[row[label_col]].update(words)

# Show top distinctive words per label
for label in sorted(label_words.keys()):
    print(f'\n{label} - Top 15 words:')
    for word, count in label_words[label].most_common(15):
        print(f'  {word}: {count:,}')

In [ ]:
# Visualize top words per label
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, label in zip(axes, sorted(label_words.keys())):
    top = label_words[label].most_common(10)
    words, counts = zip(*top)
    ax.barh(range(len(words)), counts, color=sns.color_palette('Set2')[list(sorted(label_words.keys())).index(label)])
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Count')

plt.suptitle('Top 10 Words per Label (excluding stopwords)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/label_word_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sample Examples

In [ ]:
# Show sample sentences for each label
for label in sorted(train_df[label_col].unique()):
    print(f'\n{"="*60}')
    print(f'Label: {label}')
    print('='*60)
    samples = train_df[train_df[label_col] == label].sample(3, random_state=42)
    for _, row in samples.iterrows():
        print(f'  - {row["text"][:150]}...' if len(row['text']) > 150 else f'  - {row["text"]}')

## 7. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [len(train_df), len(val_df), len(test_df)],
    'Avg Words': [train_df['word_count'].mean(), 
                  val_df['text'].apply(lambda x: len(x.split())).mean(),
                  test_df['text'].apply(lambda x: len(x.split())).mean()],
}).round(1)

print('Dataset Summary:')
print(summary.to_string(index=False))
print(f'\nTotal vocabulary (min_freq=2): {sum(1 for c in word_freq.values() if c >= 2):,}')
print(f'Number of classes: 5')
print(f'Task: Sentence-level classification of medical abstract sections')

## Key Findings

1. **Dataset Size**: The PubMed RCT 20k dataset provides a substantial training set for biomedical sentence classification.
2. **Label Distribution**: The dataset shows class imbalance, with RESULTS and METHODS being more frequent than OBJECTIVE and CONCLUSIONS.
3. **Sentence Length**: Most sentences are under 50 words; a max_length of 128 tokens covers >99% of sentences.
4. **Vocabulary**: The dataset contains a rich biomedical vocabulary. Using min_freq=2 provides a good vocabulary-coverage tradeoff.
5. **Label-Specific Patterns**: Each label has distinctive word patterns (e.g., "study" for BACKGROUND, "significant" for RESULTS), suggesting that both simple and complex models should be able to learn meaningful features.